# AIS from Global Fishing Watch for each Sentinel-1 pass

For every pass in the local archive (`<S1_DATA_DIR>/index.csv`) this notebook downloads GFW's **AIS vessel presence** for the pass area (+ `GFW_MARGIN_KM`) from `GFW_WINDOW_HOURS` before to after the pass: per vessel, per hour, on GFW's 0.01° grid (~1 km).
It writes into each pass folder:

| File | Contents |
|---|---|
| `ais_gfw.csv` | every vessel-hour in the window: vessel id, MMSI, name, flag, type, IMO, call sign, UTC hour, lat/lon of the grid cell, hours in that cell, slow (<2 kn) flag |
| `ais_at_pass.csv` | one row per vessel: its position in the hour of the pass (or the nearest hour), `hours_from_pass`, and whether it is inside the area |
| `gfw_sar.csv` | only with `GFW_WITH_SAR=true`: GFW's own radar detections for this pass, with `matched` = AIS / no AIS, a reference to compare your detector against |
| `ais_gfw.json` | request details, counts and caveats |

**Caveats**
* GFW does not provide raw AIS messages: one position per vessel per hour, snapped to ~1 km cells, and
  you don't know the minute. A ship at 15 kn moves ~28 km per hour, so only slow or anchored vessels
  (`slow_lt_2kn = True`) are reliably placed at the moment of the image.
* GFW lags about 4 days (radar detections ~5). Newer passes are skipped; run this notebook again later.
* Free for non-commercial use; attribution to Global Fishing Watch is required.

## Settings
`None` means "use `.env`".

In [ ]:
ENV_FILE = None       # path to .env; None = look in this folder (and its parent folders)
FOLDERS = None        # e.g. ["data/sentinel1/rotterdam/2026-10-28_0544"]; None = every pass in index.csv
AREAS = None          # e.g. ["rotterdam"]; None = all areas
WINDOW_HOURS = None   # hours before/after each pass; None = GFW_WINDOW_HOURS (default 2)
MARGIN_KM = None      # extra margin around each area; None = GFW_MARGIN_KM (default 10)
WITH_SAR = None       # True/False: also fetch GFW radar detections; None = GFW_WITH_SAR
FORCE = False         # True = download again even if ais_gfw.csv already exists

## Imports and `.env`

In [ ]:
import csv
import io
import json
import math
import os
import time
import zipfile
from datetime import datetime, timedelta, timezone

import requests

API = "https://gateway.api.globalfishingwatch.org/v3"
PRESENCE = "public-global-presence:latest"
SAR = "public-global-sar-presence:latest"
PRESENCE_LAG_H = 96          # GFW AIS presence is complete ~4 days after the fact
SAR_LAG_H = 120              # GFW radar detections ~5 days
DEFAULT_DATA_DIR = os.path.join("data", "sentinel1")
SLOW_FILTER = "speed = '<2'"

CAVEATS = [
    "GFW AIS presence: one position per vessel per hour, snapped to a 0.01 degree grid cell (~1 km); "
    "the minute within the hour is unknown.",
    "Moving vessels can be many km from their listed cell at the exact image time; only slow_lt_2kn vessels "
    "are reliably placed.",
    "Vessels without AIS, or whose AIS was not received, are absent.",
    "Source: Global Fishing Watch (globalfishingwatch.org). Non-commercial use only; attribution required.",
]


class StopRun(Exception):
    """A problem that needs your attention."""


def find_env(explicit=None):
    if explicit:
        return explicit
    folder = os.getcwd()
    for _ in range(3):
        p = os.path.join(folder, ".env")
        if os.path.isfile(p):
            return p
        folder = os.path.dirname(folder)
    return None


def load_env(path):
    values = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[7:]
            key, val = line.split("=", 1)
            val = val.strip()
            if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
                val = val[1:-1]
            elif " #" in val:
                val = val.split(" #", 1)[0].strip()
            values[key.strip()] = val
    return values


def settings(env_file=None):
    path = find_env(env_file)
    if path is None:
        raise StopRun("No .env found. Copy .env.example to .env and fill in GFW_TOKEN.")
    env = load_env(path)
    if not env.get("GFW_TOKEN"):
        raise StopRun(f"{path} is missing a value for: GFW_TOKEN  (https://globalfishingwatch.org/our-apis/tokens)")
    env["_path"] = path
    return env


def truthy(v):
    return str(v).strip().lower() in ("1", "true", "yes", "on")

## Helpers

In [ ]:
def parse_iso(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


def iso(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_hour(s):
    s = str(s).strip().replace("T", " ").replace("Z", "")
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d %H"):
        try:
            return datetime.strptime(s[:19], fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    return None


def grow_bbox(bbox, km):
    lat_mid = (bbox[1] + bbox[3]) / 2
    dlat, dlon = km / 111.32, km / (111.32 * math.cos(math.radians(lat_mid)))
    return [bbox[0] - dlon, bbox[1] - dlat, bbox[2] + dlon, bbox[3] + dlat]


def polygon(bbox):
    w, s, e, n = bbox
    return {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry": {
        "type": "Polygon", "coordinates": [[[w, s], [e, s], [e, n], [w, n], [w, s]]]}}]}


def write_csv(path, rows, first=()):
    keys = list(first)
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys or ["empty"])
        w.writeheader()
        w.writerows(rows)


def pick(row, *names):
    for n in names:
        if row.get(n) not in (None, ""):
            return row[n]
    return ""

## GFW client
GFW's report endpoint allows one report per token at a time, can take longer than its gateway waits,
and sometimes answers with a zip file. This client handles all of that.

In [ ]:
class GFW:
    def __init__(self, token):
        self.s = requests.Session()
        self.s.headers.update({"Authorization": f"Bearer {token}", "Content-Type": "application/json"})
        self.geojson_as_string = False

    def _err(self, r):
        try:
            return json.dumps(r.json())[:800]
        except Exception:
            return r.text[:800]

    @staticmethod
    def _rows(content, as_json=None):
        if content[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                names = [n for n in z.namelist() if n.lower().endswith(".json")]
                if not names:
                    raise RuntimeError(f"zipped report without JSON: {z.namelist()}")
                data = json.loads(z.read(max(names, key=lambda n: z.getinfo(n).file_size)))
        else:
            data = as_json if as_json is not None else json.loads(content)
        rows = []
        for entry in data.get("entries", []):
            if isinstance(entry, dict) and entry and all(isinstance(v, list) for v in entry.values()):
                for items in entry.values():
                    rows.extend(items)
            else:
                rows.append(entry)
        return rows

    def _last_report(self, max_wait_s=900):
        print("     report still running at GFW, waiting for it ...")
        t0 = time.time()
        while time.time() - t0 < max_wait_s:
            r = self.s.get(f"{API}/4wings/last-report", timeout=120)
            if r.status_code == 429:
                time.sleep(15)
                continue
            if r.status_code == 404:
                raise RuntimeError("last-report expired; run again")
            if not r.ok:
                raise RuntimeError(f"GFW {r.status_code}: {self._err(r)}")
            if r.content[:2] == b"PK":
                return self._rows(r.content)
            d = r.json()
            if d.get("status") == "running":
                time.sleep(10)
                continue
            if "entries" in d:
                return self._rows(r.content, d)
            raise RuntimeError(f"report failed: {json.dumps(d)[:600]}")
        raise RuntimeError("gave up waiting for GFW report")

    def report(self, dataset, bbox, day0, day1, group_by=None, filt=None):
        """4Wings report over a custom polygon, HIGH resolution, HOURLY. day1 is exclusive."""
        params = {"datasets[0]": dataset, "spatial-resolution": "HIGH", "temporal-resolution": "HOURLY",
                  "date-range": f"{day0:%Y-%m-%d},{day1:%Y-%m-%d}", "format": "JSON"}
        if group_by:
            params["group-by"] = group_by
        if filt:
            params["filters[0]"] = filt
        geo = polygon(bbox)
        for attempt in range(25):
            body = {"geojson": json.dumps(geo) if self.geojson_as_string else geo}
            try:
                r = self.s.post(f"{API}/4wings/report", params=params, json=body, timeout=130)
            except requests.exceptions.Timeout:
                return self._last_report()
            if r.status_code == 429:
                time.sleep(15)
                continue
            if r.status_code in (502, 504, 524):
                return self._last_report()
            if r.status_code == 503 and attempt < 6:
                time.sleep(10 * (attempt + 1))
                continue
            if r.status_code == 422 and not self.geojson_as_string and "geojson" in r.text.lower():
                self.geojson_as_string = True     # some API versions want the polygon as a JSON string
                continue
            if not r.ok:
                raise RuntimeError(f"GFW {r.status_code}: {self._err(r)}")
            try:
                as_json = r.json()
            except ValueError:
                as_json = None
            return self._rows(r.content, as_json)
        raise RuntimeError("too many retries on the GFW report endpoint")

## One pass

In [ ]:
USED_KEYS = {"date", "lat", "lon", "hours", "vesselId", "vessel_id", "id", "mmsi", "ssvid", "shipName",
             "ship_name", "name", "flag", "vesselType", "vessel_type", "geartype", "imo", "callsign"}
FIRST_COLS = ["vessel_id", "mmsi", "ship_name", "flag", "vessel_type", "imo", "callsign", "utc_hour",
              "lat", "lon", "hours_in_cell", "slow_lt_2kn", "in_area"]


def normalise(row, area_bbox, slow_keys):
    hour = parse_hour(row.get("date"))
    lat, lon = float(row.get("lat")), float(row.get("lon"))
    vid = pick(row, "vesselId", "vessel_id", "id")
    out = {
        "vessel_id": vid, "mmsi": pick(row, "mmsi", "ssvid"), "ship_name": pick(row, "shipName", "ship_name", "name"),
        "flag": pick(row, "flag"), "vessel_type": pick(row, "vesselType", "vessel_type", "geartype"),
        "imo": pick(row, "imo"), "callsign": pick(row, "callsign"),
        "utc_hour": iso(hour) if hour else row.get("date"), "lat": lat, "lon": lon,
        "hours_in_cell": row.get("hours", ""),
        "slow_lt_2kn": "" if slow_keys is None else ((vid, row.get("date"), lat, lon) in slow_keys),
        "in_area": area_bbox[0] <= lon <= area_bbox[2] and area_bbox[1] <= lat <= area_bbox[3],
    }
    for k, v in row.items():                       # keep anything else GFW returns
        if k not in USED_KEYS and not isinstance(v, (list, dict)):
            out[f"gfw_{k}"] = v
    return out, hour


def at_pass(rows_with_hours, t_pass):
    """Per vessel: the row in the pass hour (largest hours_in_cell), else the nearest hour."""
    pass_hour = t_pass.replace(minute=0, second=0, microsecond=0)
    best = {}
    for row, hour in rows_with_hours:
        if hour is None:
            continue
        key = row["vessel_id"] or row["mmsi"]
        try:
            weight = float(row["hours_in_cell"] or 0)
        except ValueError:
            weight = 0
        score = (abs((hour - pass_hour).total_seconds()) / 3600, -weight)
        if key not in best or score < best[key][0]:
            best[key] = (score, row, hour)
    out = []
    for _score, row, hour in best.values():
        r = dict(row)
        r["hours_from_pass"] = round(((hour + timedelta(minutes=30)) - t_pass).total_seconds() / 3600, 2)
        out.append(r)
    return sorted(out, key=lambda r: (abs(r["hours_from_pass"]), not r["in_area"]))


def do_pass(api, folder, area, bbox, t, window_h, margin_km, with_sar, state):
    query_bbox = grow_bbox(bbox, margin_km)
    w0, w1 = t - timedelta(hours=window_h), t + timedelta(hours=window_h)
    day0 = w0.replace(hour=0, minute=0, second=0, microsecond=0)
    day1 = w1.replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)

    print("     AIS presence per vessel ...")
    rows = api.report(PRESENCE, query_bbox, day0, day1, group_by="VESSEL_ID")
    slow_keys = None
    if state.get("slow_filter_ok", True):
        print("     slow vessels (<2 kn) ...")
        try:
            slow = api.report(PRESENCE, query_bbox, day0, day1, group_by="VESSEL_ID", filt=SLOW_FILTER)
            slow_keys = {(pick(r, "vesselId", "vessel_id", "id"), r.get("date"), float(r["lat"]), float(r["lon"]))
                         for r in slow}
        except RuntimeError as e:
            if "422" in str(e) or "400" in str(e):
                print("     ⚠️ GFW did not accept the speed filter; slow_lt_2kn left empty from now on")
                state["slow_filter_ok"] = False
            else:
                raise

    norm = []
    for r in rows:
        try:
            n, hour = normalise(r, bbox, slow_keys)
        except (TypeError, ValueError):
            continue
        if hour is None or (w0 - timedelta(hours=1)) < hour <= w1:
            norm.append((n, hour))
    norm.sort(key=lambda nh: (nh[0]["utc_hour"], nh[0]["vessel_id"]))
    all_rows = [n for n, _ in norm]
    pass_rows = at_pass(norm, t)
    write_csv(os.path.join(folder, "ais_gfw.csv"), all_rows, FIRST_COLS)
    write_csv(os.path.join(folder, "ais_at_pass.csv"), pass_rows, FIRST_COLS + ["hours_from_pass"])

    sar_counts = None
    if with_sar:
        if datetime.now(timezone.utc) - t < timedelta(hours=SAR_LAG_H):
            print("     GFW radar detections not available yet for this pass (~5 day lag); skipped")
        else:
            sar_rows = []
            pass_hour = t.replace(minute=0, second=0, microsecond=0)
            for matched in ("false", "true"):
                print(f"     GFW radar detections, matched={matched} ...")
                for r in api.report(SAR, query_bbox, day0, day1, filt=f"matched='{matched}'"):
                    hour = parse_hour(r.get("date"))
                    if hour and abs((hour - pass_hour).total_seconds()) <= 3600:
                        sar_rows.append({"utc_hour": iso(hour), "lat": r.get("lat"), "lon": r.get("lon"),
                                         "detections": r.get("detections"), "matched": matched == "true",
                                         "in_area": bbox[0] <= float(r["lon"]) <= bbox[2]
                                         and bbox[1] <= float(r["lat"]) <= bbox[3]})
            write_csv(os.path.join(folder, "gfw_sar.csv"), sar_rows,
                      ["utc_hour", "lat", "lon", "detections", "matched", "in_area"])
            sar_counts = {"unmatched": sum(not r["matched"] for r in sar_rows),
                          "matched": sum(r["matched"] for r in sar_rows)}

    meta = {
        "area": area, "pass_utc": iso(t), "area_bbox": bbox, "query_bbox": [round(v, 5) for v in query_bbox],
        "window_utc": [iso(w0), iso(w1)], "dataset": PRESENCE, "grid": "0.01 degree, hourly, per vessel",
        "rows": len(all_rows), "vessels": len({r["vessel_id"] or r["mmsi"] for r in all_rows}),
        "vessels_at_pass_in_area": sum(1 for r in pass_rows if r["in_area"] and abs(r["hours_from_pass"]) <= 1),
        "slow_flag": slow_keys is not None, "gfw_sar": sar_counts,
        "downloaded_utc": iso(datetime.now(timezone.utc)), "caveats": CAVEATS,
    }
    with open(os.path.join(folder, "ais_gfw.json"), "w") as f:
        json.dump(meta, f, indent=2)
    return meta

## All passes

In [ ]:
def load_passes(data_dir, only_folders, only_areas):
    passes = []
    if only_folders:
        for folder in only_folders:
            with open(os.path.join(folder, "scene.json")) as f:
                m = json.load(f)
            passes.append((os.path.abspath(folder), m["area"], m["bbox_lonlat"], parse_iso(m["acquired_utc"])))
        return passes
    index = os.path.join(data_dir, "index.csv")
    if not os.path.exists(index):
        raise StopRun(f"No {index}. Run 01_s1_download.ipynb first.")
    with open(index, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    known = sorted({r["area"] for r in rows})
    if only_areas and not set(only_areas) & set(known):
        raise StopRun(f"No passes for area(s) {only_areas}; the archive has: {', '.join(known) or 'nothing yet'}")
    for r in rows:
        if only_areas and r["area"] not in only_areas:
            continue
        passes.append((os.path.join(data_dir, r["folder"]), r["area"],
                       [float(v) for v in r["bbox"].split()], parse_iso(r["acquired_utc"])))
    return passes


def download_ais(env_file=None, folders=None, areas=None, window_hours=None, margin_km=None, with_sar=None,
                 force=False):
    env = settings(env_file)
    data_dir = os.path.abspath(env.get("S1_DATA_DIR") or DEFAULT_DATA_DIR)
    window_h = float(window_hours if window_hours is not None else (env.get("GFW_WINDOW_HOURS") or 2))
    margin = float(margin_km if margin_km is not None else (env.get("GFW_MARGIN_KM") or 10))
    with_sar = truthy(env.get("GFW_WITH_SAR", "false")) if with_sar is None else bool(with_sar)
    passes = load_passes(data_dir, folders, areas)
    print(f"Settings: {env['_path']}\nArchive:  {data_dir}  ({len(passes)} pass(es))\n"
          f"Window:   +/-{window_h:g} h around each pass, area + {margin:g} km"
          f"{', with GFW radar detections' if with_sar else ''}")

    api, state, now = GFW(env["GFW_TOKEN"]), {}, datetime.now(timezone.utc)
    done, skipped, waiting, failed = [], 0, 0, 0
    for folder, area, bbox, t in sorted(passes, key=lambda x: x[3]):
        name = f"{area} {t:%Y-%m-%d %H:%M} UTC"
        if os.path.exists(os.path.join(folder, "ais_gfw.csv")) and not force:
            skipped += 1
            continue
        ready = t + timedelta(hours=window_h + PRESENCE_LAG_H)
        if now < ready:
            print(f"  {name}: GFW AIS not complete yet; run again after {ready:%Y-%m-%d %H:%M} UTC")
            waiting += 1
            continue
        print(f"  {name}")
        try:
            m = do_pass(api, folder, area, bbox, t, window_h, margin, with_sar, state)
            sar = (f", GFW radar: {m['gfw_sar']['unmatched']} unmatched / {m['gfw_sar']['matched']} matched"
                   if m["gfw_sar"] else "")
            print(f"     -> {m['vessels']} vessels, {m['vessels_at_pass_in_area']} in the area at pass time{sar}")
            done.append(folder)
        except RuntimeError as e:
            print(f"     FAILED: {e}")
            failed += 1
    print(f"\nAIS downloaded for {len(done)} pass(es); {skipped} already had it; {waiting} too recent; {failed} failed.")
    print("Data: Global Fishing Watch (globalfishingwatch.org) - non-commercial use, attribution required.")
    return {"data_dir": data_dir, "passes": [p[0] for p in passes], "done": done}

## Run

In [ ]:
try:
    result = download_ais(ENV_FILE, FOLDERS, AREAS, WINDOW_HOURS, MARGIN_KM, WITH_SAR, FORCE)
except StopRun as e:
    result = None
    print("⚠️ ", e)

## Look at the result
A summary per pass, and the vessels at pass time for the newest pass with AIS
(inside the area first, nearest in time first).

In [ ]:
from IPython.display import display

if result:
    summary = []
    for folder in result["passes"]:
        meta_path = os.path.join(folder, "ais_gfw.json")
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                m = json.load(f)
            summary.append({"area": m["area"], "pass_utc": m["pass_utc"], "vessels": m["vessels"],
                            "in_area_at_pass": m["vessels_at_pass_in_area"], "slow_flag": m["slow_flag"],
                            "gfw_radar": m["gfw_sar"], "folder": folder})
    try:
        import pandas as pd
        display(pd.DataFrame(summary))
        if summary:
            newest = max(summary, key=lambda s: s["pass_utc"])
            print(f"\nVessels at pass time: {newest['area']} {newest['pass_utc']}")
            display(pd.read_csv(os.path.join(newest["folder"], "ais_at_pass.csv")).head(25))
    except ImportError:
        for s in summary:
            print(s)